In [263]:
# ============================================================================
# IMPORT CÁC THƯ VIỆN CẦN THIẾT CHO XAI-RL FRAMEWORK
# ============================================================================
from pathlib import Path
import sys

# Nếu notebook đang nằm trong: SARSA_FinancialRL/application/EIDT_Project/
PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
import os
sys.path.append('d:\\NCKH\\SARSA_FinancialRL')

# 2. Data Processing
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 3. Deep Learning - PyTorch
import torch
from torch import nn
import torch.nn.functional as F

# 4. Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style cho plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 5. XAI Libraries - CHỈ DÙNG SHAP
try:
    import shap
    print("✓ SHAP available")
except ImportError:
    print("⚠ SHAP not installed - will install when needed")

# 6. Project-specific Imports
from agents.d_sarsa.d_sarsa import Qsa
from environments.stock_trading_env.mdp import StockTradingMDP
from data.data_provider.library_extracted.vnstock.VNStockDataProvider import VNStockDataProvider
from data.data_processor.feature_engineer import engineer_stat as es

# 7. Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("="*80)
print("✓ Tất cả thư viện đã được import thành công!")
print("="*80)
print("\n🎯 XAI-RL Framework - 3 Phương pháp độc lập:")
print("  [1] RDX  - Reward Decomposition (weights từ domain knowledge)")
print("  [2] MSX  - Multi-Step Explanation (trajectory analysis)")
print("  [3] SHAP - Feature Attribution (Shapley values)")
print("\n📊 Deep RL Agent:")
print("  • Qsa:              Q-network (input=7, output=11)")
print("  • StockTradingMDP:  Environment cho stock trading")
print("  • VNStockData:      Data provider cho VN market")
print("\nReady to analyze SARSA agent! 🚀")
print("="*80)

Project root: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL
⚠ SHAP not installed - will install when needed
✓ Tất cả thư viện đã được import thành công!

🎯 XAI-RL Framework - 3 Phương pháp độc lập:
  [1] RDX  - Reward Decomposition (weights từ domain knowledge)
  [2] MSX  - Multi-Step Explanation (trajectory analysis)
  [3] SHAP - Feature Attribution (Shapley values)

📊 Deep RL Agent:
  • Qsa:              Q-network (input=7, output=11)
  • StockTradingMDP:  Environment cho stock trading
  • VNStockData:      Data provider cho VN market

Ready to analyze SARSA agent! 🚀


In [264]:
# 1.1. Load model SARSA
qsa = Qsa(input_size=7, num_classes=11)

from pathlib import Path
import os
import torch

MODELS_DIR = PROJECT_ROOT / "models"
EIDT_MODELS_DIR = MODELS_DIR / "EIDT"

symbols = ["ACB", "FPT", "GAS", "HPG", "SSI", "VCB"]


def get_model_paths_greedy_bad(model_folder):
    model_folder = Path(model_folder)

    return {
        symbol: model_folder / f"{symbol}_bad_sarsa_ucb_vae_best.pth"
        for symbol in symbols
    }


# Chọn folder con bạn muốn chạy RDX
current_eidt_folder = EIDT_MODELS_DIR / "UCB_BAD"

model_paths = get_model_paths_greedy_bad(current_eidt_folder)

model_path_acb_phase_1 = str(model_paths["ACB"])
model_path_fpt_phase_1 = str(model_paths["FPT"])
model_path_gas_phase_1 = str(model_paths["GAS"])
model_path_hpg_phase_1 = str(model_paths["HPG"])
model_path_ssi_phase_1 = str(model_paths["SSI"])
model_path_vcb_phase_1 = str(model_paths["VCB"])


from collections import OrderedDict
import os
import torch

if os.path.exists(model_path_acb_phase_1):
    state_dict = torch.load(model_path_acb_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_acb_phase_1}")
else:
    print(f"✗ Model not found: {model_path_acb_phase_1}")


# # 1.2. Load dữ liệu 
# provider = VNStockDataProvider()
# print("\nĐang lấy dữ liệu từ vnstock...")
# df_raw = provider.get_ohlcv_data('SSI', '2021-12-14', '2024-12-31')
# print(f"✓ Đã lấy {len(df_raw)} dòng dữ liệu")

# # 1.3. Xử lý dữ liệu và thêm technical indicators
# df_processed = df_raw.copy()
# df_processed.rename(columns={'date': 'time'}, inplace=True)
# df_processed['time'] = pd.to_datetime(df_processed['time']).dt.strftime('%d/%m/%Y')
# df_processed = es.add_technical_indicators(df_processed, start_date= '01/01/2022')
# print(f"✓ Đã thêm technical indicators: {df_processed.shape}")

from pathlib import Path
import pandas as pd

# Nếu notebook/script đang chạy từ thư mục gốc project: SARSA_FinancialRL
def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "SARSA_FinancialRL", *cwd.parents]

    for p in candidates:
        if (p / "data").exists() and (p / "agents").exists() and (p / "environments").exists():
            return p

    raise FileNotFoundError("Không tìm thấy project root chứa data/, agents/, environments/")


PROJECT_ROOT = find_project_root()
base_dir = PROJECT_ROOT / "data" / "data_storer" / "data_research" / "test"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("base_dir:", base_dir)
print("base_dir exists:", base_dir.exists())

df_processed_test_ACB = pd.read_csv(base_dir / "test_ACB_phase_2.csv")
df_processed_test_ACB.rename(columns={"date": "time"}, inplace=True)
df_processed_test_ACB["time"] = pd.to_datetime(df_processed_test_ACB["time"]).dt.strftime("%d/%m/%Y")

df_processed_test_FPT = pd.read_csv(base_dir / "test_FPT_phase_2.csv")
df_processed_test_FPT.rename(columns={"date": "time"}, inplace=True)
df_processed_test_FPT["time"] = pd.to_datetime(df_processed_test_FPT["time"]).dt.strftime("%d/%m/%Y")

df_processed_test_GAS = pd.read_csv(base_dir / "test_GAS_phase_2.csv")
df_processed_test_GAS.rename(columns={"date": "time"}, inplace=True)
df_processed_test_GAS["time"] = pd.to_datetime(df_processed_test_GAS["time"]).dt.strftime("%d/%m/%Y")

df_processed_test_HPG = pd.read_csv(base_dir / "bad_test_HPG.csv")
df_processed_test_HPG.rename(columns={"date": "time"}, inplace=True)
df_processed_test_HPG["time"] = pd.to_datetime(df_processed_test_HPG["time"]).dt.strftime("%d/%m/%Y")

df_processed_test_SSI = pd.read_csv(base_dir / "test_SSI_phase_2.csv")
df_processed_test_SSI.rename(columns={"date": "time"}, inplace=True)
df_processed_test_SSI["time"] = pd.to_datetime(df_processed_test_SSI["time"]).dt.strftime("%d/%m/%Y")

df_processed_test_VCB = pd.read_csv(base_dir / "test_VCB_phase_2.csv")
df_processed_test_VCB.rename(columns={"date": "time"}, inplace=True)
df_processed_test_VCB["time"] = pd.to_datetime(df_processed_test_VCB["time"]).dt.strftime("%d/%m/%Y")


# Ghép dữ liệu: train trước rồi đến test (theo thời gian) thay vì dùng toán tử '+'
# df_processed = pd.concat([df_processed_train, df_processed_test], ignore_index=True)
# Đảm bảo thứ tự thời gian tăng dần nếu chưa chắc chắn
# _df_time = pd.to_datetime(df_processed['time'], format='%d/%m/%Y')
# df_processed = df_processed.assign(_time=_df_time).sort_values('_time').drop(columns=['_time']).reset_index(drop=True)
# print(f"✓ Merged train+test: {df_processed.shape} (train={df_processed_train.shape}, test={df_processed_test.shape})")
# print(f"  Time range: {df_processed['time'].iloc[0]} -> {df_processed['time'].iloc[-1]}")


✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\ACB_bad_sarsa_ucb_vae_best.pth
PROJECT_ROOT: D:\nckh\SARSA_FinancialRL\SARSA_FinancialRL
base_dir: D:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\data\data_storer\data_research\test
base_dir exists: True


In [265]:
df_processed_test_HPG

,time,open,high,low,close,volume,MACD,RSI,CCI,ADX
0,04/01/2022,26.83,27.06,26.72,26.80,20100700,-0.565777,45.550621,27.225506,30.678471
1,05/01/2022,26.89,27.15,26.83,26.83,18760700,-0.500479,45.963412,41.610376,28.991851
2,06/01/2022,26.66,26.77,26.43,26.43,17172700,-0.475525,41.451122,-24.640400,28.079455
3,07/01/2022,26.43,26.49,26.17,26.26,16335300,-0.464116,39.668716,-62.769038,27.618181
4,10/01/2022,26.29,26.75,26.09,26.17,17477500,-0.457068,38.719472,-52.410123,26.625678
...,...,...,...,...,...,...,...,...,...,...
493,25/12/2023,20.57,20.91,20.57,20.79,20414974,0.157600,55.944903,40.875912,14.095142
494,26/12/2023,20.76,21.14,20.76,21.06,33753533,0.183383,59.460872,98.654481,14.647102
495,27/12/2023,21.10,21.21,21.02,21.02,19860170,0.198303,58.713277,115.260144,15.310545
496,28/12/2023,21.02,21.36,20.95,21.17,31321948,0.219699,60.708317,126.289308,16.245551


In [266]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_ACB = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_ACB = df_processed_test_ACB.iloc[0]
state_init_ACB = [
    float(first_row_ACB['close']),
    mdp_ACB.balance_init,
    0,
    float(first_row_ACB['MACD']),
    float(first_row_ACB['RSI']),
    float(first_row_ACB['CCI']),
    float(first_row_ACB['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_ACB, rewards_ACB, actions_ACB = mdp_ACB.simulate(
    df_processed_test_ACB[1:].reset_index(drop=True), 
    state_init_ACB, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_ACB)} states, {len(actions_ACB)} actions")
print(f"  Total reward: {sum(rewards_ACB):.2f}")
print(f"  Final portfolio: ${states_ACB[-1][1] + states_ACB[-1][0]*states_ACB[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: -18.07
  Final portfolio: $981.93


In [267]:
if os.path.exists(model_path_fpt_phase_1):
    state_dict = torch.load(model_path_fpt_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_fpt_phase_1}")
else:
    print(f"✗ Model not found: {model_path_fpt_phase_1}")


✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\FPT_bad_sarsa_ucb_vae_best.pth


In [268]:
# 1.5. Khởi tạo MDP và chạy simulation
mdp_FPT = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_FPT = df_processed_test_FPT.iloc[0]
state_init_FPT = [
    float(first_row_FPT['close']),
    mdp_FPT.balance_init,
    0,
    float(first_row_FPT['MACD']),
    float(first_row_FPT['RSI']),
    float(first_row_FPT['CCI']),
    float(first_row_FPT['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_FPT, rewards_FPT, actions_FPT = mdp_FPT.simulate(
    df_processed_test_FPT[1:].reset_index(drop=True), 
    state_init_FPT, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_FPT)} states, {len(actions_FPT)} actions")
print(f"  Total reward: {sum(rewards_FPT):.2f}")
print(f"  Final portfolio: ${states_FPT[-1][1] + states_FPT[-1][0]*states_FPT[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: 529.83
  Final portfolio: $1529.83


In [269]:
if os.path.exists(model_path_gas_phase_1):
    state_dict = torch.load(model_path_gas_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_gas_phase_1}")
else:
    print(f"✗ Model not found: {model_path_gas_phase_1}")


✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\GAS_bad_sarsa_ucb_vae_best.pth


In [270]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_GAS = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_GAS = df_processed_test_GAS.iloc[0]
state_init_GAS = [
    float(first_row_GAS['close']),
    mdp_GAS.balance_init,
    0,
    float(first_row_GAS['MACD']),
    float(first_row_GAS['RSI']),
    float(first_row_GAS['CCI']),
    float(first_row_GAS['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_GAS, rewards_GAS, actions_GAS = mdp_GAS.simulate(
    df_processed_test_GAS[1:].reset_index(drop=True), 
    state_init_GAS, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_GAS)} states, {len(actions_GAS)} actions")
print(f"  Total reward: {sum(rewards_GAS):.2f}")
print(f"  Final portfolio: ${states_GAS[-1][1] + states_GAS[-1][0]*states_GAS[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: 82.45
  Final portfolio: $1082.45


In [271]:
if os.path.exists(model_path_hpg_phase_1):
    state_dict = torch.load(model_path_hpg_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_hpg_phase_1}")
else:
    print(f"✗ Model not found: {model_path_hpg_phase_1}")


✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\HPG_bad_sarsa_ucb_vae_best.pth


In [272]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_HPG = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_HPG = df_processed_test_HPG.iloc[0]
state_init_HPG = [
    float(first_row_HPG['close']),
    mdp_HPG.balance_init,
    0,
    float(first_row_HPG['MACD']),
    float(first_row_HPG['RSI']),
    float(first_row_HPG['CCI']),
    float(first_row_HPG['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_HPG, rewards_HPG, actions_HPG = mdp_HPG.simulate(
    df_processed_test_HPG[1:].reset_index(drop=True), 
    state_init_HPG, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_HPG)} states, {len(actions_HPG)} actions")
print(f"  Total reward: {sum(rewards_HPG):.2f}")
print(f"  Final portfolio: ${states_HPG[-1][1] + states_HPG[-1][0]*states_HPG[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 498 states, 497 actions
  Total reward: 60.07
  Final portfolio: $1060.07


In [273]:
if os.path.exists(model_path_ssi_phase_1):
    state_dict = torch.load(model_path_ssi_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_ssi_phase_1}")
else:
    print(f"✗ Model not found: {model_path_ssi_phase_1}")


✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\SSI_bad_sarsa_ucb_vae_best.pth


In [274]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_SSI = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_SSI = df_processed_test_SSI.iloc[0]
state_init_SSI = [
    float(first_row_SSI['close']),
    mdp_SSI.balance_init,
    0,
    float(first_row_SSI['MACD']),
    float(first_row_SSI['RSI']),
    float(first_row_SSI['CCI']),
    float(first_row_SSI['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_SSI, rewards_SSI, actions_SSI = mdp_SSI.simulate(
    df_processed_test_SSI[1:].reset_index(drop=True), 
    state_init_SSI, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_SSI)} states, {len(actions_SSI)} actions")
print(f"  Total reward: {sum(rewards_SSI):.2f}")
print(f"  Final portfolio: ${states_SSI[-1][1] + states_SSI[-1][0]*states_SSI[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: 0.00
  Final portfolio: $1000.00


In [275]:
if os.path.exists(model_path_vcb_phase_1):
    state_dict = torch.load(model_path_vcb_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_vcb_phase_1}")
else:
    print(f"✗ Model not found: {model_path_vcb_phase_1}")


✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\VCB_bad_sarsa_ucb_vae_best.pth


In [276]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_VCB = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_VCB = df_processed_test_VCB.iloc[0]
state_init_VCB = [
    float(first_row_VCB['close']),
    mdp_VCB.balance_init,
    0,
    float(first_row_VCB['MACD']),
    float(first_row_VCB['RSI']),
    float(first_row_VCB['CCI']),
    float(first_row_VCB['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_VCB, rewards_VCB, actions_VCB = mdp_VCB.simulate(
    df_processed_test_VCB[1:].reset_index(drop=True), 
    state_init_VCB, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_VCB)} states, {len(actions_VCB)} actions")
print(f"  Total reward: {sum(rewards_VCB):.2f}")
print(f"  Final portfolio: ${states_VCB[-1][1] + states_VCB[-1][0]*states_VCB[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: 597.02
  Final portfolio: $1597.02


In [277]:
import numpy as np
import torch

def run_simulation_and_decompose_q_balanced(mdp, model, df_test, state_init, w_vector=[1.0, 0.5, 1.0, 0.1], alpha=0.005):
    """
    Phiên bản Cân Bằng (Balanced): 
    Chuẩn hóa Q-total theo % Tài sản để Profit ngang hàng với Risk/Stab.
    """
    # 1. TÍNH TOÁN CÁC CHỈ SỐ KỸ THUẬT (Vectorized)
    prices = df_test['close'].values
    returns = np.zeros_like(prices)
    safe_prices = np.where(prices[:-1] == 0, 1e-9, prices[:-1]) 
    returns[1:] = (prices[1:] - prices[:-1]) / safe_prices
    
    running_max = np.maximum.accumulate(prices)
    MA10 = np.convolve(prices, np.ones(10)/10, mode='same')
    
    # 2. KHỞI TẠO
    states = [state_init]
    actions = []
    rewards = []
    q_value_history = [] 
    
    w1, w2, w3, w4 = w_vector
    current_state = state_init
    
    print(f"Đang chạy simulation (Balanced Mode)...")
    
    # 3. VÒNG LẶP CHÍNH
    for t in range(len(df_test) - 1):
        
        # --- A. LẤY Q-TOTAL TỪ MODEL (Đơn vị: Tiền tích lũy dự kiến) ---
        state_tensor = torch.FloatTensor(current_state)
        with torch.no_grad():
            q_total_scalar = model(state_tensor).squeeze().cpu().numpy()
            
        # --- B. TÍNH TỔNG TÀI SẢN HIỆN TẠI ĐỂ CHUẨN HÓA ---
        # Portfolio = Tiền mặt + (Số cổ phiếu * Giá hiện tại)
        current_portfolio = current_state[1] + (current_state[2] * prices[t])
        
        # Tránh chia cho 0 hoặc số quá nhỏ
        if current_portfolio < 10: current_portfolio = 1000 
            
        # --- C. PHÂN RÃ Q-VALUE (ĐÃ CHUẨN HÓA VỀ %) ---
        q_matrix_t = np.zeros((11, 4))
        
        # Tính toán các chỉ số môi trường
        current_price = prices[t]
        current_dd = (current_price - running_max[t]) / running_max[t] if running_max[t] != 0 else 0
        current_return = returns[t]
        
        trend_val = current_price - MA10[t]
        sign_trend = np.sign(trend_val)
        
        for i in range(11):
            action_val = i - 5 
            
            # --- 1. CHUẨN HÓA Q TỔNG VỀ % (QUAN TRỌNG NHẤT) ---
            # Q ($16.36) / Portfolio ($1000) = 0.01636 (1.6%)
            # Lúc này 0.016 (Profit) đã ngang hàng với 0.025 (Pos)
            q_total_norm = q_total_scalar[i] / current_portfolio
            
            # --- 2. TÍNH CÁC THÀNH PHẦN PHỤ (VỐN ĐÃ LÀ %) ---
            # Risk: % Drawdown
            val_risk = w2 * -abs(current_dd)          
            
            # Stability: % Biến động ngày
            val_stab = w4 * -abs(current_return)      
            
            # Position: % Thưởng Trend
            sign_action = np.sign(action_val)
            if action_val == 0:
                val_pos = 0
            elif sign_action == sign_trend:
                val_pos = w3 * alpha
            else:
                val_pos = w3 * -alpha
                
            # --- 3. TÍNH PROFIT (PHẦN DƯ SAU KHI ĐÃ CHUẨN HÓA) ---
            # Profit (%) = Tổng (%) - (Risk% + Pos% + Stab%)
            val_profit = q_total_norm - (val_risk + val_pos + val_stab)
            
            q_matrix_t[i] = [val_profit, val_risk, val_pos, val_stab]
            
        q_value_history.append(q_matrix_t)
        
        # --- D. CHỌN HÀNH ĐỘNG & BƯỚC ĐI TIẾP THEO ---
        # Lưu ý: Agent vẫn chọn dựa trên Q gốc (Tiền) để đảm bảo đúng logic đã học
        action_idx = np.argmax(q_total_scalar) 
        real_action = action_idx - 5
        
        next_row = df_test.iloc[t + 1]
        next_state, reward, done = mdp.step(current_state, real_action, next_row)
        
        states.append(next_state) 
        actions.append(real_action)
        rewards.append(reward)
        
        current_state = next_state
        if done:
            break
            
    return states, actions, rewards, q_value_history

## Mã giả (Pseudocode): Thuật toán Phân Rã Q-Value (RDX)

```
ALGORITHM: Run_Simulation_And_Decompose_Q_Balanced
INPUT:
    - mdp: Môi trường MDP (Stock Trading Environment)
    - model: Neural network đã train (Q-network)
    - df_test: DataFrame dữ liệu test (giá cổ phiếu, indicators)
    - state_init: Trạng thái khởi tạo [price, balance, position, MACD, RSI, CCI, ADX]
    - w_vector: Vector trọng số [w₁, w₂, w₃, w₄] = [1.0, 0.5, 1.0, 0.1]
    - alpha: Hệ số thưởng/phạt cho Position (mặc định = 0.005)

OUTPUT:
    - states: Lịch sử các trạng thái
    - actions: Lịch sử hành động
    - rewards: Lịch sử rewards
    - q_value_history: Lịch sử Q-matrix (11 x 4) tại mỗi timestep

─────────────────────────────────────────────────────────────────────────
PHASE 1: TIỀN XỬ LÝ DỮ LIỆU (VECTORIZED)
─────────────────────────────────────────────────────────────────────────

prices ← df_test['close'].values
n ← length(prices)

// Tính returns (% thay đổi giá)
FOR t = 2 TO n:
    returns[t] ← (prices[t] - prices[t-1]) / prices[t-1]
END FOR

// Tính Running Maximum (để xác định Drawdown)
running_max[1] ← prices[1]
FOR t = 2 TO n:
    running_max[t] ← max(running_max[t-1], prices[t])
END FOR

// Tính Moving Average 10 ngày
MA10 ← moving_average(prices, window=10)

─────────────────────────────────────────────────────────────────────────
PHASE 2: KHỞI TẠO
─────────────────────────────────────────────────────────────────────────

states ← [state_init]
actions ← []
rewards ← []
q_value_history ← []

current_state ← state_init
w₁, w₂, w₃, w₄ ← w_vector  // [Profit, Risk, Position, Stability]

─────────────────────────────────────────────────────────────────────────
PHASE 3: VÒNG LẶP CHÍNH - MÔ PHỎNG & PHÂN RÃ
─────────────────────────────────────────────────────────────────────────

FOR t = 1 TO (n - 1):
    
    // ══════════════════════════════════════════════════════════════
    // BƯỚC A: DỰ ĐOÁN Q-TOTAL TỪ MODEL (Đơn vị: Tiền tệ $)
    // ══════════════════════════════════════════════════════════════
    
    state_tensor ← convert_to_tensor(current_state)
    q_total_money ← model.forward(state_tensor)  // Shape: (11,)
    
    // ══════════════════════════════════════════════════════════════
    // BƯỚC B: TÍNH TỔNG TÀI SẢN ĐỂ CHUẨN HÓA VỀ %
    // ══════════════════════════════════════════════════════════════
    
    cash ← current_state[1]
    position ← current_state[2]
    current_price ← prices[t]
    
    // Portfolio Value = Cash + (Position × Price)
    current_portfolio ← cash + (position × current_price)
    
    IF current_portfolio < 10 THEN
        current_portfolio ← 1000  // Ngăn chia cho 0
    END IF
    
    // ══════════════════════════════════════════════════════════════
    // BƯỚC C: PHÂN RÃ Q-VALUE CHO TẤT CẢ 11 HÀNH ĐỘNG [-5, +5]
    // ══════════════════════════════════════════════════════════════
    
    q_matrix_t ← zeros(11, 4)  // Ma trận 11 hành động × 4 thành phần
    
    // Tính các chỉ số môi trường tại timestep t
    current_dd ← (current_price - running_max[t]) / running_max[t]
    current_return ← returns[t]
    trend_direction ← sign(current_price - MA10[t])
    
    FOR action_index = 0 TO 10:
        action_value ← action_index - 5  // Map [0,10] → [-5,+5]
        
        // ─────────────────────────────────────────────────────────
        // Bước C.1: CHUẨN HÓA Q-TOTAL VỀ % TÀI SẢN
        // ─────────────────────────────────────────────────────────
        // Ví dụ: Q($16.36) / Portfolio($1000) = 0.01636 (1.636%)
        
        q_total_percent ← q_total_money[action_index] / current_portfolio
        
        // ─────────────────────────────────────────────────────────
        // Bước C.2: TÍNH CÁC THÀNH PHẦN PHỤ (ĐÃ LÀ %)
        // ─────────────────────────────────────────────────────────
        
        // 1. RISK COMPONENT (% Drawdown)
        //    - Giá trị âm → Rủi ro cao
        //    - Drawdown càng sâu → Risk component càng âm
        risk_value ← w₂ × (-|current_dd|)
        
        // 2. STABILITY COMPONENT (% Volatility)
        //    - Giá trị âm → Biến động cao
        //    - Return càng lớn → Stability càng âm
        stability_value ← w₄ × (-|current_return|)
        
        // 3. POSITION COMPONENT (% Trend Alignment Bonus)
        //    - Dương: Hành động cùng chiều với trend
        //    - Âm: Hành động ngược chiều với trend
        action_direction ← sign(action_value)
        
        IF action_value = 0 THEN
            position_value ← 0  // Hold không có bonus/penalty
        ELSE IF action_direction = trend_direction THEN
            position_value ← w₃ × alpha  // Thưởng khi cùng chiều
        ELSE
            position_value ← w₃ × (-alpha)  // Phạt khi ngược chiều
        END IF
        
        // ─────────────────────────────────────────────────────────
        // Bước C.3: TÍNH PROFIT (PHẦN DƯ - RESIDUAL)
        // ─────────────────────────────────────────────────────────
        // Profit = Tổng (%) - (Risk% + Position% + Stability%)
        
        profit_value ← q_total_percent - (risk_value + position_value + stability_value)
        
        // Lưu vào ma trận Q
        q_matrix_t[action_index] ← [profit_value, risk_value, position_value, stability_value]
        
    END FOR
    
    // Lưu ma trận Q đã phân rã vào lịch sử
    q_value_history.append(q_matrix_t)
    
    // ══════════════════════════════════════════════════════════════
    // BƯỚC D: CHỌN HÀNH ĐỘNG & THỰC THI TRONG MÔI TRƯỜNG
    // ══════════════════════════════════════════════════════════════
    
    // Chọn hành động tốt nhất dựa trên Q gốc (Tiền $)
    best_action_idx ← argmax(q_total_money)
    real_action ← best_action_idx - 5
    
    // Thực thi hành động trong MDP
    next_row ← df_test.row[t + 1]
    next_state, reward, done ← mdp.step(current_state, real_action, next_row)
    
    // Cập nhật lịch sử
    states.append(next_state)
    actions.append(real_action)
    rewards.append(reward)
    
    current_state ← next_state
    
    IF done THEN
        BREAK
    END IF
    
END FOR

RETURN states, actions, rewards, q_value_history

─────────────────────────────────────────────────────────────────────────
CÔNG THỨC TOÁN HỌC CHI TIẾT
─────────────────────────────────────────────────────────────────────────

Cho hành động a ∈ {-5, -4, ..., +4, +5}, tại timestep t:

1. Q-Total (Raw from Model):
   Q_total(s_t, a) = Neural_Network(s_t)[a]  [Đơn vị: $]

2. Portfolio Normalization:
   Portfolio_t = Balance_t + Position_t × Price_t
   Q_norm(s_t, a) = Q_total(s_t, a) / Portfolio_t  [Đơn vị: %]

3. Component Decomposition:
   • Risk:      R(s_t) = w₂ × (-|DD_t|)  where DD_t = (Price_t - Max_t) / Max_t
   • Stability: S(s_t) = w₄ × (-|Return_t|)
   • Position:  P(s_t, a) = w₃ × α × sign(a) × sign(Trend_t)
   • Profit:    π(s_t, a) = Q_norm(s_t, a) - [R(s_t) + S(s_t) + P(s_t, a)]

4. Reconstruction Property (Residual Approach):
   Q_norm(s_t, a) = π(s_t, a) + R(s_t) + P(s_t, a) + S(s_t)
   
   ⟹ Σ components = Q_total / Portfolio  ✓

─────────────────────────────────────────────────────────────────────────
ĐẶC ĐIỂM QUAN TRỌNG
─────────────────────────────────────────────────────────────────────────

1. NORMALIZATION KEY INSIGHT:
   - Q gốc ($16) / Portfolio ($1000) = 1.6%
   - Profit (%) giờ ngang hàng với Risk (%), Stability (%)
   - Tránh Profit bị lấn át khi tính bằng đồng tiền tuyệt đối

2. RESIDUAL DESIGN:
   - Profit = Phần còn lại sau khi trừ đi các thành phần đã biết
   - Đảm bảo tổng các thành phần = Q-Total chuẩn hóa

3. DOMAIN KNOWLEDGE:
   - Risk: Drawdown (Market risk)
   - Stability: Volatility (Uncertainty)
   - Position: Trend alignment (Tactical advantage)
   - Profit: Expected return (Core objective)

4. COMPLEXITY: O(n × k) where n = timesteps, k = 11 actions
```

In [278]:
# 1. Chạy lại Simulation với hàm Balanced
print("Đang tính toán lại Q-values (Chuẩn hóa %)...")
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_ACB, 
    qsa, 
    df_processed_test_ACB, 
    state_init_ACB,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_ACB = states_new
actions_ACB = actions_new
rewards_ACB = rewards_new
q_values_history_ACB = q_values_history_new
# Kỳ vọng: Profit giờ chỉ khoảng 0.016 (1.6%) thay vì 16.3

Đang tính toán lại Q-values (Chuẩn hóa %)...
Đang chạy simulation (Balanced Mode)...


In [279]:
if os.path.exists(model_path_fpt_phase_1):
    state_dict = torch.load(model_path_fpt_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_fpt_phase_1}")
else:
    print(f"✗ Model not found: {model_path_fpt_phase_1}")


"""if os.path.exists(model_path_fpt_phase_1):
    state_dict = torch.load(model_path_fpt_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_fpt_phase_1}")
else:
    print(f"✗ Model not found: {model_path_fpt_phase_1}")
    """

✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\FPT_bad_sarsa_ucb_vae_best.pth


'if os.path.exists(model_path_fpt_phase_1):\n    state_dict = torch.load(model_path_fpt_phase_1, map_location=torch.device(\'cpu\'))\n    qsa.load_state_dict(state_dict)\n    qsa.eval()\n    print(f"✓ Model loaded: {model_path_fpt_phase_1}")\nelse:\n    print(f"✗ Model not found: {model_path_fpt_phase_1}")\n    '

In [280]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_FPT, 
    qsa, 
    df_processed_test_FPT, 
    state_init_FPT,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_FPT = states_new
actions_FPT = actions_new
rewards_FPT = rewards_new
q_values_history_FPT = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [281]:
if os.path.exists(model_path_gas_phase_1):
    state_dict = torch.load(model_path_gas_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_gas_phase_1}")
else:
    print(f"✗ Model not found: {model_path_gas_phase_1}")




"""
if os.path.exists(model_path_gas_phase_1):
    state_dict = torch.load(model_path_gas_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_gas_phase_1}")
else:
    print(f"✗ Model not found: {model_path_gas_phase_1}")
    """

✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\GAS_bad_sarsa_ucb_vae_best.pth


'\nif os.path.exists(model_path_gas_phase_1):\n    state_dict = torch.load(model_path_gas_phase_1, map_location=torch.device(\'cpu\'))\n    qsa.load_state_dict(state_dict)\n    qsa.eval()\n    print(f"✓ Model loaded: {model_path_gas_phase_1}")\nelse:\n    print(f"✗ Model not found: {model_path_gas_phase_1}")\n    '

In [282]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_GAS, 
    qsa, 
    df_processed_test_GAS, 
    state_init_GAS,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_GAS = states_new
actions_GAS = actions_new
rewards_GAS = rewards_new
q_values_history_GAS = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [283]:
if os.path.exists(model_path_hpg_phase_1):
    state_dict = torch.load(model_path_hpg_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_hpg_phase_1}")
else:
    print(f"✗ Model not found: {model_path_hpg_phase_1}")





"""
if os.path.exists(model_path_hpg_phase_1):
    state_dict = torch.load(model_path_hpg_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_hpg_phase_1}")
else:
    print(f"✗ Model not found: {model_path_hpg_phase_1}")
    """

✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\HPG_bad_sarsa_ucb_vae_best.pth


'\nif os.path.exists(model_path_hpg_phase_1):\n    state_dict = torch.load(model_path_hpg_phase_1, map_location=torch.device(\'cpu\'))\n    qsa.load_state_dict(state_dict)\n    qsa.eval()\n    print(f"✓ Model loaded: {model_path_hpg_phase_1}")\nelse:\n    print(f"✗ Model not found: {model_path_hpg_phase_1}")\n    '

In [284]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_HPG, 
    qsa, 
    df_processed_test_HPG, 
    state_init_HPG,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_HPG = states_new
actions_HPG = actions_new
rewards_HPG = rewards_new
q_values_history_HPG = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [285]:
if os.path.exists(model_path_ssi_phase_1):
    state_dict = torch.load(model_path_ssi_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_ssi_phase_1}")
else:
    print(f"✗ Model not found: {model_path_ssi_phase_1}")


"""
if os.path.exists(model_path_ssi_phase_1):
    state_dict = torch.load(model_path_ssi_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_ssi_phase_1}")
else:
    print(f"✗ Model not found: {model_path_ssi_phase_1}")
    """

✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\SSI_bad_sarsa_ucb_vae_best.pth


'\nif os.path.exists(model_path_ssi_phase_1):\n    state_dict = torch.load(model_path_ssi_phase_1, map_location=torch.device(\'cpu\'))\n    qsa.load_state_dict(state_dict)\n    qsa.eval()\n    print(f"✓ Model loaded: {model_path_ssi_phase_1}")\nelse:\n    print(f"✗ Model not found: {model_path_ssi_phase_1}")\n    '

In [286]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_SSI, 
    qsa, 
    df_processed_test_SSI, 
    state_init_SSI,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_SSI = states_new
actions_SSI = actions_new
rewards_SSI = rewards_new
q_values_history_SSI = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [287]:
if os.path.exists(model_path_vcb_phase_1):
    state_dict = torch.load(model_path_vcb_phase_1, map_location=torch.device("cpu"))

    # Model Greedy_bad lưu theo tên layer "net",
    # còn class Qsa hiện tại dùng tên layer "fc_liner".
    fixed_state_dict = OrderedDict()
    for key, value in state_dict.items():
        new_key = key.replace("net.", "fc_liner.")
        fixed_state_dict[new_key] = value

    qsa.load_state_dict(fixed_state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_vcb_phase_1}")
else:
    print(f"✗ Model not found: {model_path_vcb_phase_1}")

"""
if os.path.exists(model_path_vcb_phase_1):
    state_dict = torch.load(model_path_vcb_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_vcb_phase_1}")
else:
    print(f"✗ Model not found: {model_path_vcb_phase_1}")
"""

✓ Model loaded: d:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\models\EIDT\UCB_BAD\VCB_bad_sarsa_ucb_vae_best.pth


'\nif os.path.exists(model_path_vcb_phase_1):\n    state_dict = torch.load(model_path_vcb_phase_1, map_location=torch.device(\'cpu\'))\n    qsa.load_state_dict(state_dict)\n    qsa.eval()\n    print(f"✓ Model loaded: {model_path_vcb_phase_1}")\nelse:\n    print(f"✗ Model not found: {model_path_vcb_phase_1}")\n'

In [288]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_VCB, 
    qsa, 
    df_processed_test_VCB, 
    state_init_VCB,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_VCB = states_new
actions_VCB = actions_new
rewards_VCB = rewards_new
q_values_history_VCB = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [289]:
import numpy as np

def analyze_msx(q_vector_selected, q_vector_compared, component_names):
    """
    Phân tích Minimal Sufficient Explanation (MSX) cho một cặp hành động.
    
    Args:
        q_vector_selected: Vector Q của hành động được chọn (VD: Mua).
        q_vector_compared: Vector Q của hành động so sánh (VD: Giữ/Bán).
        component_names: List tên các thành phần ['Profit', 'Risk', 'Pos', 'Stab'].
        
    Returns:
        dict: Kết quả phân tích MSX.
    """
    # 1. Tính vector khác biệt (RDX - Reward Difference Explanation) [cite: 1070]
    # Delta > 0: Lý do ủng hộ hành động chọn
    # Delta < 0: Lý do phản đối (ủng hộ hành động kia)
    delta = q_vector_selected - q_vector_compared
    
    # 2. Phân loại Pros (Tích cực) và Cons (Tiêu cực)
    pros = [] 
    disadvantage_sum = 0.0 # Tổng bất lợi (d)
    
    for i, val in enumerate(delta):
        if val > 0:
            pros.append((i, val))
        else:
            disadvantage_sum += abs(val) # [cite: 1077]
            
    # 3. Sắp xếp các lý do tích cực từ lớn đến bé (Greedy approach) [cite: 1082]
    pros.sort(key=lambda x: x[1], reverse=True)
    
    # 4. Tìm tập hợp MSX+ (Minimal Sufficient Set) [cite: 1079]
    msx_plus = []
    current_sum = 0.0
    is_sufficient = False
    
    # Cộng dồn lý do tốt nhất cho đến khi thắng được disadvantage
    for idx, val in pros:
        msx_plus.append({
            'component': component_names[idx],
            'value': val,
            'contribution_percent': 0 # Sẽ tính sau
        })
        current_sum += val
        
        if current_sum > disadvantage_sum:
            is_sufficient = True
            break
            
    # Tính % đóng góp của từng lý do trong tập MSX
    for item in msx_plus:
        item['contribution_percent'] = (item['value'] / current_sum) * 100

    # 5. Xác định các lý do tiêu cực chính (Optional - để hiển thị Disadvantage)
    cons = []
    for i, val in enumerate(delta):
        if val < 0:
            cons.append({'component': component_names[i], 'value': val})

    return {
        'is_dominated': (disadvantage_sum == 0), # Nếu không có bất lợi nào [cite: 1093]
        'msx_plus': msx_plus,
        'disadvantage': disadvantage_sum,
        'cons_details': cons,
        'full_delta': delta
    }

In [290]:
import numpy as np
import pandas as pd
from scipy.signal import argrelextrema

def identify_critical_points(df_test, actions_history, q_values_history=None, action_change_threshold=3, trend_window=10, top_k=15):
    """
    Lọc ra Top K điểm quan trọng nhất từ kết quả chạy Test.
    
    Args:
        top_k (int): Số lượng điểm tối đa muốn lấy (mặc định 15).
    """
    prices = df_test['close'].values
    actions = np.array(actions_history)
    n = len(prices)
    
    # --- GIAI ĐOẠN 1: TÌM TẤT CẢ ỨNG VIÊN (CANDIDATES) ---
    
    # 1. Nhóm Đáy/DD (Priority 1 - Quan trọng nhất)
    idxs_bottom = []
    global_min_idx = np.argmin(prices)
    idxs_bottom.append(global_min_idx)
    
    # Tìm Max Drawdown
    running_max = np.maximum.accumulate(prices)
    # Tránh chia cho 0
    safe_running_max = np.where(running_max == 0, 1e-9, running_max)
    drawdowns = (prices - running_max) / safe_running_max
    max_dd_idx = np.argmin(drawdowns)
    
    if max_dd_idx != global_min_idx:
        idxs_bottom.append(max_dd_idx)
    
    # 2. Nhóm Đảo chiều (Priority 2)
    peaks = argrelextrema(prices, np.greater, order=trend_window)[0]
    valleys = argrelextrema(prices, np.less, order=trend_window)[0]
    idxs_reversal = np.concatenate((peaks, valleys))
    
    # Chấm điểm đảo chiều: Ưu tiên điểm có giá lệch xa nhất so với Median
    median_price = np.median(prices)
    # Tạo list (index, score) và sort giảm dần theo score (độ lệch)
    scored_reversals = [(idx, abs(prices[idx] - median_price)) for idx in idxs_reversal]
    scored_reversals.sort(key=lambda x: x[1], reverse=True)
    sorted_reversal_indices = [x[0] for x in scored_reversals]

    # 3. Nhóm Thay đổi hành động (Priority 3)
    action_diffs = np.abs(actions[1:] - actions[:-1])
    # Lấy index (cộng 1 vì diff làm lệch index)
    shift_candidates = np.where(action_diffs >= action_change_threshold)[0] + 1
    
    # Chấm điểm shift: Ưu tiên cú quay xe gắt nhất (Magnitude lớn nhất)
    scored_shifts = [(idx, action_diffs[idx-1]) for idx in shift_candidates]
    scored_shifts.sort(key=lambda x: x[1], reverse=True)
    sorted_shift_indices = [x[0] for x in scored_shifts]

    # --- GIAI ĐOẠN 2: CHỌN LỌC (FILLING) ---
    final_indices = set()
    
    # Bước 1: Lấy tất cả Đáy/DD
    for idx in idxs_bottom:
        if len(final_indices) < top_k:
            final_indices.add(idx)
            
    # Bước 2: Lấy các điểm Đảo chiều lớn (Điền tiếp vào chỗ trống)
    for idx in sorted_reversal_indices:
        if len(final_indices) < top_k:
            final_indices.add(idx)
            
    # Bước 3: Nếu vẫn chưa đủ, lấy thêm các Action Shift lớn nhất
    for idx in sorted_shift_indices:
        if len(final_indices) < top_k:
            final_indices.add(idx)
            
    # --- GIAI ĐOẠN 3: TỔNG HỢP VÀ PHÂN LOẠI LẠI ---
    sorted_unique_indices = sorted(list(final_indices))
    
    # Mapping lại vào dictionary để biết điểm nào thuộc loại nào
    critical_points = {
        'trend_reversal': [],
        'lowest_price': [],
        'action_shift': []
    }
    
    # Dùng set để lookup cho nhanh
    set_bottoms = set(idxs_bottom)
    set_reversals = set(idxs_reversal)
    set_shifts = set(shift_candidates)
    
    for idx in sorted_unique_indices:
        # Một điểm có thể thuộc nhiều loại
        if idx in set_bottoms:
            critical_points['lowest_price'].append(idx)
        if idx in set_reversals:
            critical_points['trend_reversal'].append(idx)
        if idx in set_shifts:
            critical_points['action_shift'].append(idx)

    print(f"Đã lọc Top {len(sorted_unique_indices)} điểm quan trọng (Limit: {top_k}).")
    print(f"- Đáy/DD sâu: {len(critical_points['lowest_price'])}")
    print(f"- Đảo chiều lớn: {len(critical_points['trend_reversal'])}")
    print(f"- Đổi Action gắt: {len(critical_points['action_shift'])}")
    
    return critical_points, sorted_unique_indices

## Mã giả (Pseudocode): Thuật toán Lọc Critical Points

```
ALGORITHM: Identify_Critical_Points
INPUT: 
    - prices: Chuỗi giá đóng cửa [p₁, p₂, ..., pₙ]
    - actions: Chuỗi hành động của agent [a₁, a₂, ..., aₙ]
    - action_change_threshold: Ngưỡng thay đổi hành động (mặc định = 3)
    - trend_window: Cửa sổ phát hiện đảo chiều (mặc định = 10)
    - top_k: Số điểm quan trọng cần lấy (mặc định = 15)

OUTPUT:
    - critical_points: Dictionary chứa 3 loại điểm {lowest_price, trend_reversal, action_shift}
    - sorted_unique_indices: List các index đã sắp xếp

─────────────────────────────────────────────────────────────────────────
PHASE 1: TÌM CÁC ỨNG VIÊN (CANDIDATES DETECTION)
─────────────────────────────────────────────────────────────────────────

1. NHÓM ĐÁY/DRAWDOWN (Priority 1 - Cao nhất):
   candidates_bottom = []
   
   // Tìm điểm giá thấp nhất toàn chuỗi
   global_min_idx ← argmin(prices)
   candidates_bottom.add(global_min_idx)
   
   // Tìm điểm Drawdown lớn nhất
   running_max ← cumulative_max(prices)
   drawdowns ← (prices - running_max) / running_max
   max_dd_idx ← argmin(drawdowns)
   
   IF max_dd_idx ≠ global_min_idx THEN
       candidates_bottom.add(max_dd_idx)
   END IF

2. NHÓM ĐẢO CHIỀU GIẢX (Priority 2):
   // Tìm đỉnh và đáy cục bộ
   peaks ← find_local_maxima(prices, window=trend_window)
   valleys ← find_local_minima(prices, window=trend_window)
   candidates_reversal ← peaks ∪ valleys
   
   // Chấm điểm và sắp xếp theo độ lệch so với median
   median_price ← median(prices)
   FOR EACH idx IN candidates_reversal:
       score[idx] ← |prices[idx] - median_price|
   END FOR
   
   sorted_reversal ← sort(candidates_reversal, by=score, descending=True)

3. NHÓM THAY ĐỔI HÀNH ĐỘNG (Priority 3):
   // Tìm các điểm có sự thay đổi hành động lớn
   action_diffs ← |actions[t] - actions[t-1]| for t = 2 to n
   candidates_shift ← {t | action_diffs[t] ≥ action_change_threshold}
   
   // Chấm điểm và sắp xếp theo độ lớn thay đổi
   FOR EACH idx IN candidates_shift:
       score[idx] ← action_diffs[idx]
   END FOR
   
   sorted_shift ← sort(candidates_shift, by=score, descending=True)

─────────────────────────────────────────────────────────────────────────
PHASE 2: CHỌN LỌC TOP-K (GREEDY SELECTION)
─────────────────────────────────────────────────────────────────────────

final_indices ← empty_set()

// Bước 1: Lấy TẤT CẢ điểm Đáy/DD (Priority cao nhất)
FOR EACH idx IN candidates_bottom:
    IF |final_indices| < top_k THEN
        final_indices.add(idx)
    END IF
END FOR

// Bước 2: Điền thêm các điểm Đảo chiều lớn (theo thứ tự đã sort)
FOR EACH idx IN sorted_reversal:
    IF |final_indices| < top_k THEN
        final_indices.add(idx)
    END IF
END FOR

// Bước 3: Nếu chưa đủ, điền các Action Shift lớn
FOR EACH idx IN sorted_shift:
    IF |final_indices| < top_k THEN
        final_indices.add(idx)
    END IF
END FOR

─────────────────────────────────────────────────────────────────────────
PHASE 3: PHÂN LOẠI VÀ TỔ CHỨC KẾT QUẢ
─────────────────────────────────────────────────────────────────────────

// Sắp xếp theo thứ tự thời gian
sorted_unique_indices ← sort(final_indices)

// Phân loại từng điểm vào các nhóm (1 điểm có thể thuộc nhiều nhóm)
critical_points ← {
    'lowest_price': [],
    'trend_reversal': [],
    'action_shift': []
}

FOR EACH idx IN sorted_unique_indices:
    IF idx IN candidates_bottom THEN
        critical_points['lowest_price'].add(idx)
    END IF
    
    IF idx IN candidates_reversal THEN
        critical_points['trend_reversal'].add(idx)
    END IF
    
    IF idx IN candidates_shift THEN
        critical_points['action_shift'].add(idx)
    END IF
END FOR

RETURN critical_points, sorted_unique_indices

─────────────────────────────────────────────────────────────────────────
COMPLEXITY ANALYSIS:
─────────────────────────────────────────────────────────────────────────
- Time: O(n log n) - do sorting các candidates
- Space: O(n) - lưu trữ các candidates và scores
  where n = số lượng timesteps trong chuỗi dữ liệu
```

### Giải thích Logic:

**Ưu tiên phân cấp (Priority Hierarchy):**
1. **Đáy/DD** - Quan trọng nhất (luôn được chọn trước)
2. **Đảo chiều** - Quan trọng thứ hai (điểm có biên độ lớn nhất được ưu tiên)
3. **Thay đổi hành động** - Điền vào chỗ trống còn lại

**Đặc điểm:**
- Greedy: Lấy dần từng điểm cho đến khi đủ `top_k`
- Non-overlapping: Dùng `set()` để tránh trùng lặp
- Multi-label: Một điểm có thể thuộc nhiều loại cùng lúc

In [291]:
critical_dict_ACB, index_list_ACB = identify_critical_points(
    df_processed_test_ACB, 
    actions_ACB, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 1
- Đảo chiều lớn: 14
- Đổi Action gắt: 4


In [292]:
critical_dict_FPT, index_list_FPT = identify_critical_points(
    df_processed_test_FPT, 
    actions_FPT, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 12 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 2
- Đảo chiều lớn: 12
- Đổi Action gắt: 0


In [293]:
critical_dict_GAS, index_list_GAS = identify_critical_points(
    df_processed_test_GAS, 
    actions_GAS, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 2
- Đảo chiều lớn: 14
- Đổi Action gắt: 8


In [294]:
critical_dict_HPG, index_list_HPG = identify_critical_points(
    df_processed_test_HPG, 
    actions_HPG, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 1
- Đảo chiều lớn: 15
- Đổi Action gắt: 3


In [295]:
critical_dict_SSI, index_list_SSI = identify_critical_points(
    df_processed_test_SSI, 
    actions_SSI, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 11 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 1
- Đảo chiều lớn: 10
- Đổi Action gắt: 0


In [296]:
critical_dict_VCB, index_list_VCB = identify_critical_points(
    df_processed_test_VCB, 
    actions_VCB, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 2
- Đảo chiều lớn: 10
- Đổi Action gắt: 5


In [297]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_ACB:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_ACB.iloc[t]['close']
    date = df_processed_test_ACB.iloc[t]['time']
    actual_action = actions_ACB[t]
    actual_reward = rewards_ACB[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_ACB.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_ACB.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_ACB.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_ACB[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=1 (03/01/2019) - Giá: 6.24
>> Loại điểm: [ĐẢO CHIỀU]
>> Hành động thực tế: +0
>> Reward thực tế nhận được: 0.000000

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +0] vs [Bỏ: Mua Mạnh (+5)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.103536 |       -0.035289 |       +0.138824
   Risk       |       -0.009259 |       -0.009259 |       +0.000000
   Pos        |        0.000000 |        0.001250 |       -0.001250
   Stab       |       -0.009259 |       -0.009259 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.085017 |       -0.052557 |       +0.137574
   
   [GIẢI THÍCH MSX]:
      -> Chọn vì lý do ch

In [298]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_FPT:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_FPT.iloc[t]['close']
    date = df_processed_test_FPT.iloc[t]['time']
    actual_action = actions_FPT[t]
    actual_reward = rewards_FPT[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_FPT.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_FPT.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_FPT.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_FPT[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=1 (03/01/2019) - Giá: 12.88
>> Loại điểm: [ĐẢO CHIỀU | ĐÁY/DD]
>> Hành động thực tế: +1
>> Reward thực tế nhận được: 0.187120

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +1] vs [Bỏ: Giữ (0)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.301370 |        0.116657 |       +0.184713
   Risk       |       -0.002498 |       -0.002498 |       +0.000000
   Pos        |        0.001250 |        0.000000 |       +0.001250
   Stab       |       -0.002498 |       -0.002498 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.297624 |        0.111660 |       +0.185963
   
   [GIẢI THÍCH MSX]:
      -> Quyết định c

In [299]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_GAS:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_GAS.iloc[t]['close']
    date = df_processed_test_GAS.iloc[t]['time']
    actual_action = actions_GAS[t]
    actual_reward = rewards_GAS[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_GAS.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_GAS.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_GAS.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_GAS[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=2 (04/01/2019) - Giá: 50.19
>> Loại điểm: [ĐẢO CHIỀU | ĐÁY/DD]
>> Hành động thực tế: +2
>> Reward thực tế nhận được: 6.319620

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +2] vs [Bỏ: Giữ (0)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.081746 |        0.001679 |       +0.080067
   Risk       |       -0.006596 |       -0.006596 |       +0.000000
   Pos        |        0.001250 |        0.000000 |       +0.001250
   Stab       |       -0.000596 |       -0.000596 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.075804 |       -0.005513 |       +0.081317
   
   [GIẢI THÍCH MSX]:
      -> Quyết định c

IndexError: list index out of range

In [ ]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_HPG:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_HPG.iloc[t]['close']
    date = df_processed_test_HPG.iloc[t]['time']
    actual_action = actions_HPG[t]
    actual_reward = rewards_HPG[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_HPG.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_HPG.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_HPG.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_HPG[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=2 (05/01/2017) - Giá: 5.22
>> Loại điểm: [ĐẢO CHIỀU]
>> Hành động thực tế: +5
>> Reward thực tế nhận được: -1.376100

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +5] vs [Bỏ: Giữ (0)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.135374 |        0.090277 |       +0.045097
   Risk       |       -0.000000 |       -0.000000 |       +0.000000
   Pos        |        0.001250 |        0.000000 |       +0.001250
   Stab       |       -0.000962 |       -0.000962 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.135662 |        0.089315 |       +0.046347
   
   [GIẢI THÍCH MSX]:
      -> Quyết định chọn 5 tốt

In [ ]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_SSI:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_SSI.iloc[t]['close']
    date = df_processed_test_SSI.iloc[t]['time']
    actual_action = actions_SSI[t]
    actual_reward = rewards_SSI[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_SSI.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_SSI.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_SSI.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_SSI[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=14 (23/01/2017) - Giá: 6.43
>> Loại điểm: [ĐẢO CHIỀU | ĐÁY/DD]
>> Hành động thực tế: +5
>> Reward thực tế nhận được: 1.467850

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +5] vs [Bỏ: Giữ (0)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.101212 |        0.056607 |       +0.044605
   Risk       |       -0.007541 |       -0.007541 |       +0.000000
   Pos        |       -0.001250 |        0.000000 |       -0.001250
   Stab       |       -0.001929 |       -0.001929 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.090491 |        0.047136 |       +0.043355
   
   [GIẢI THÍCH MSX]:
      -> Chọn vì lý d

In [ ]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_VCB:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_VCB.iloc[t]['close']
    date = df_processed_test_VCB.iloc[t]['time']
    actual_action = actions_VCB[t]
    actual_reward = rewards_VCB[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_VCB.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_VCB.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_VCB.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_VCB[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=1 (04/01/2017) - Giá: 15.30
>> Loại điểm: [ĐẢO CHIỀU]
>> Hành động thực tế: +0
>> Reward thực tế nhận được: 0.000000

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +0] vs [Bỏ: Mua Mạnh (+5)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.258951 |        0.168201 |       +0.090750
   Risk       |       -0.001785 |       -0.001785 |       +0.000000
   Pos        |        0.000000 |        0.001250 |       -0.001250
   Stab       |       -0.001785 |       -0.001785 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.255382 |        0.165882 |       +0.089500
   
   [GIẢI THÍCH MSX]:
      -> Chọn vì lý do c

In [ ]:
'''
import sys
!{sys.executable} -m pip uninstall -y kaleido
!{sys.executable} -m pip install kaleido==0.2.1
'''

'\nimport sys\n!{sys.executable} -m pip uninstall -y kaleido\n!{sys.executable} -m pip install kaleido==0.2.1\n'

In [ ]:
import plotly
import plotly.io as pio
import importlib.metadata as metadata
print("plotly renderer:", pio.renderers.default)

def plot_rdx_components_at_time_t(agents_data, time_index=0, components=['Profit', 'Risk', 'Pos', 'Stab'], scale=50):
    """
    Vẽ plot RDX tại 1 thời điểm t cụ thể, hiển thị 4 thành phần của hành động đã chọn.
    Mỗi agent có 4 cột (Profit, Risk, Pos, Stab) đặt cạnh nhau.
    
    Args:
        agents_data: dict chứa thông tin của từng agent
        time_index: int - chỉ số trong danh sách critical_indices (0 = điểm đầu tiên)
        components: list tên các thành phần ['Profit', 'Risk', 'Pos', 'Stab']
        scale: hệ số scale để hiển thị
    """
    import plotly.graph_objects as go
    
    colors = {
        'Profit': '#2ecc71',  # Xanh lá
        'Risk': '#e74c3c',    # Đỏ
        'Pos': '#3498db',     # Xanh dương
        'Stab': '#f39c12'     # Cam
    }
    
    # Chuẩn bị dữ liệu
    x_agents = []  # Tên agent
    x_components = []  # Tên component
    y_values = []
    colors_list = []
    
    # Thông tin để hiển thị
    agent_actions = {}  # Lưu hành động của mỗi agent
    
    # Tìm max Q để chuẩn hóa chung
    all_q = []
    for ticker, data in agents_data.items():
        critical_indices = data['critical_indices']
        if time_index < len(critical_indices):
            t = critical_indices[time_index]
            all_q.append(data['q_history'][t])
    
    all_q = np.array(all_q)
    max_abs_q = np.max(np.abs(all_q))
    if max_abs_q == 0: max_abs_q = 1
    
    print(f"Vẽ RDX tại điểm thời gian thứ {time_index + 1} (index={time_index})")
    print(f"Max Abs Q: {max_abs_q:.4f}\n")
    
    # Duyệt qua từng agent
    for ticker, data in agents_data.items():
        q_history = data['q_history']
        actions = data['actions']
        critical_indices = data['critical_indices']
        
        # Kiểm tra xem time_index có hợp lệ không
        if time_index >= len(critical_indices):
            print(f"⚠ {ticker}: time_index={time_index} vượt quá số điểm quan trọng ({len(critical_indices)})")
            continue
        
        # Lấy thời điểm t cụ thể
        t = critical_indices[time_index]
        act = actions[t]
        agent_actions[ticker] = act
        
        # print(f"{ticker}: t={t}, action={act:+}")
        
        # Chuẩn hóa Q-matrix
        q_mat = (q_history[t] / max_abs_q) * scale
        
        # Xác định index của hành động đã chọn
        idx_selected = act + 5  # Map [-5, 5] về [0, 10]
        vec_selected = q_mat[idx_selected]
        
        # Lấy giá trị của từng component
        for comp_idx, comp_name in enumerate(components):
            x_agents.append(ticker)
            x_components.append(comp_name)
            y_values.append(vec_selected[comp_idx])
            colors_list.append(colors[comp_name])
    
    # Tạo figure với grouped bar chart
    fig = go.Figure()
    
    # Vẽ bar cho từng component
    for comp_name in components:
        x_data = []
        y_data = []
        
        for i, (agent, comp) in enumerate(zip(x_agents, x_components)):
            if comp == comp_name:
                x_data.append(agent)
                y_data.append(y_values[i])
        
        fig.add_trace(
            go.Bar(
                x=x_data,
                y=y_data,
                name=comp_name,
                marker_color=colors[comp_name],
                text=[f'{v:.2f}' for v in y_data],
                textposition='outside',
                textfont=dict(size=9)
            )
        )
    
    # Tạo subtitle hiển thị hành động của mỗi agent
    action_text = " | ".join([f"{ticker}: {act:+}" for ticker, act in agent_actions.items()])
    
    # Cập nhật layout
    fig.update_layout(
        title=dict(
            # text=f"RDX Components at Critical Point #{time_index + 1}<br><sub>Actions: {action_text}</sub>",
            x=0.5,
            xanchor='center'
        ),
        height=600,
        plot_bgcolor='white',
        barmode='group',
        bargap=0.15,
        bargroupgap=0.1,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        xaxis=dict(
            title="Agent",
            showticklabels=True,
            tickangle=0
        ),
        yaxis=dict(
            title=f"Normalized Q-Value (x{scale})",
            showgrid=True,
            gridcolor='#eee',
            zeroline=True,
            zerolinewidth=2,
            zerolinecolor='black'
        )
    )
    
    return fig

# Chuẩn bị dữ liệu cho tất cả các agent
agents_data = {
    'ACB': {
        'q_history': q_values_history_ACB,
        'actions': actions_ACB,
        'critical_indices': index_list_ACB
    },
    'FPT': {
        'q_history': q_values_history_FPT,
        'actions': actions_FPT,
        'critical_indices': index_list_FPT
    },
    'GAS': {
        'q_history': q_values_history_GAS,
        'actions': actions_GAS,
        'critical_indices': index_list_GAS
    },
    'HPG': {
        'q_history': q_values_history_HPG,
        'actions': actions_HPG,
        'critical_indices': index_list_HPG
    },
    'SSI': {
        'q_history': q_values_history_SSI,
        'actions': actions_SSI,
        'critical_indices': index_list_SSI
    },
    'VCB': {
        'q_history': q_values_history_VCB,
        'actions': actions_VCB,
        'critical_indices': index_list_VCB
    }
}

# Gọi hàm vẽ tại điểm thời gian đầu tiên (time_index=0)
# Bạn có thể thay đổi time_index=1, 2, 3... để xem các điểm khác
fig = plot_rdx_components_at_time_t(agents_data, time_index=1, scale=1)

# Xuất hình ảnh
from pathlib import Path

output_dir = PROJECT_ROOT / "Graph+Pic"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "ver2_RDX_components_comparison_t1.png"

fig.write_image(str(output_path), format="png", width=1400, height=600, scale=2)

print(f"\n✓ Đã xuất hình ảnh tại: {output_path}")


# Hiển thị figure
fig.show()

plotly renderer: vscode
Vẽ RDX tại điểm thời gian thứ 2 (index=1)
Max Abs Q: 0.2529


✓ Đã xuất hình ảnh tại: D:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\Graph+Pic\ver2_RDX_components_comparison_t1.png


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

def plot_agent_reasoning_minimalist(ticker, df_price, actions, q_history, critical_indices, scale=50):
    """
    Phiên bản Minimalist (Final + Auto Normalize):
    - Tự động chuẩn hóa Q-value về khoảng nhỏ.
    - Mũi tên sát đỉnh cột.
    """
    
    # --- BƯỚC 0: CHUẨN HÓA DỮ LIỆU ĐẦU VÀO ---
    # Gom tất cả các ma trận Q tại các điểm critical lại để tìm max
    all_critical_q = []
    for t in critical_indices:
        all_critical_q.append(q_history[t])
    
    all_critical_q = np.array(all_critical_q)
    # Tìm giá trị tuyệt đối lớn nhất để làm mẫu số
    max_abs_q = np.max(np.abs(all_critical_q))
    if max_abs_q == 0: max_abs_q = 1 
    
    print(f"Max Abs Q ban đầu: {max_abs_q:.4f}. Đang chuẩn hóa để vẽ...")

    # --- 1. CHUẨN BỊ DỮ LIỆU ---
    # Subplot 1
    buy_x, buy_y = [], []
    sell_x, sell_y = [], []
    hold_x, hold_y = [], []
    
    # Subplot 2
    x_dates_hidden = [] 
    x_actions = []      
    
    y_profit, y_risk, y_pos, y_stab = [], [], [], []
    
    selected_marker_x_grp = []
    selected_marker_x_act = []
    selected_marker_y = []
    
    tick_vals_top = []
    tick_text_top = []
    
    for i, t in enumerate(critical_indices):
        date_str = df_price.iloc[t]['time']
        price = df_price.iloc[t]['close']
        act = actions[t]
        
        # --- Data Subplot 1 ---
        tick_vals_top.append(date_str)
        tick_text_top.append(date_str) 
        
        if act > 0:
            buy_x.append(date_str); buy_y.append(price)
        elif act < 0:
            sell_x.append(date_str); sell_y.append(price)
        else:
            hold_x.append(date_str); hold_y.append(price)
            
        # --- QUAN TRỌNG: CHUẨN HÓA VÀ SCALE ---
        q_mat_raw = q_history[t]
        # Chuẩn hóa về [-1, 1] rồi nhân scale
        q_mat = (q_mat_raw / max_abs_q) * scale
        
        group_id = f"#{i+1}" 
        
        compare_set = [
            ("Sell (-5)", 0),
            ("Hold (0)", 5),
            ("Buy (+5)", 10)
        ]
        
        for label, idx in compare_set:
            vec = q_mat[idx]
            
            x_dates_hidden.append(group_id) 
            x_actions.append(label)
            
            y_profit.append(vec[0])
            y_risk.append(vec[1])
            y_pos.append(vec[2])
            y_stab.append(vec[3])
            
            is_selected = False
            if act < 0 and idx == 0: is_selected = True
            elif act == 0 and idx == 5: is_selected = True
            elif act > 0 and idx == 10: is_selected = True
            
            if is_selected:
                selected_marker_x_grp.append(group_id)
                selected_marker_x_act.append(label)
                
                # Tính vị trí sát cột (Margin 2%)
                total_pos = sum([v for v in vec if v > 0])
                if total_pos > 0:
                    marker_y = total_pos + (scale * 0.02)
                else:
                    marker_y = 0 + (scale * 0.02)
                    
                selected_marker_y.append(marker_y)

    # --- 2. VẼ BIỂU ĐỒ ---
    fig = make_subplots(
        rows=2, cols=1, 
        vertical_spacing=0.1,
        row_heights=[0.6, 0.4],
        specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
    )

    # === SUBPLOT 1: PRICE ===
    fig.add_trace(go.Scatter(x=df_price['time'], y=df_price['close'], mode='lines', name='Price', line=dict(color='black', width=1), hoverinfo='x+y'), row=1, col=1)
    
    # Markers
    fig.add_trace(go.Scatter(x=buy_x, y=buy_y, mode='markers', name='Buy', marker=dict(symbol='triangle-up', size=14, color='green', line=dict(width=1, color='black'))), row=1, col=1)
    fig.add_trace(go.Scatter(x=sell_x, y=sell_y, mode='markers', name='Sell', marker=dict(symbol='triangle-down', size=14, color='red', line=dict(width=1, color='black'))), row=1, col=1)
    fig.add_trace(go.Scatter(x=hold_x, y=hold_y, mode='markers', name='Hold', marker=dict(symbol='circle', size=10, color='#f1c40f', line=dict(width=1, color='black'))), row=1, col=1)

    # === SUBPLOT 2: STACKED BARS ===
    for name, y_data, color in zip(
        ['Profit', 'Risk', 'Trend', 'Stability'],
        [y_profit, y_risk, y_pos, y_stab],
        ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
    ):
        fig.add_trace(go.Bar(
            x=[x_dates_hidden, x_actions], 
            y=y_data, name=name, marker_color=color, legendgroup='comp'
        ), row=2, col=1)

    # Marker Selected (Mũi tên có Legend - chỉ markers, không có đường nối)
    fig.add_trace(go.Scatter(
        x=[selected_marker_x_grp, selected_marker_x_act], 
        y=selected_marker_y,
        mode='markers',  # Chỉ hiển thị markers, không có đường nối
        marker=dict(symbol='triangle-down', size=14, color='black'),
        name='Agent Selection',
        showlegend=True
    ), row=2, col=1)

    # --- 3. LAYOUT TINH CHỈNH ---
    fig.update_layout(
        height=900,
        plot_bgcolor='white',
        barmode='relative',
        bargap=0.4,       
        bargroupgap=0.05,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    # TINH CHỈNH TRỤC X1 (TRÊN)
    fig.update_xaxes(
        tickmode='array',
        tickvals=tick_vals_top,
        ticktext=tick_text_top,
        tickangle=-90, 
        showgrid=True, gridcolor='#eee',
        row=1, col=1
    )
    
    # TINH CHỈNH TRỤC X2 (DƯỚI)
    fig.update_xaxes(
        type='multicategory',
        tickangle=-90, 
        showticklabels=True,
        title_text="", 
        row=2, col=1
    )
    
    # Các trục Y
    fig.update_yaxes(title_text="Closing price", showgrid=True, gridcolor='#eee', row=1, col=1)
    # Cập nhật nhãn trục Y dưới để phản ánh việc đã chuẩn hóa
    fig.update_yaxes(title_text=f"Norm. Q-Value (x{scale})", showgrid=True, gridcolor='#eee', row=2, col=1)
    
    fig.add_hline(y=0, line_dash="solid", line_color="black", line_width=1, row=2, col=1)
    fig.update_xaxes(rangeslider_visible=False, row=1, col=1)

    # fig.show()
    return fig



In [ ]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='HPG',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_HPG,  # Dữ liệu giá
    actions=actions_HPG,             # Hành động thực tế
    q_history=q_values_history_HPG,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_HPG,     # Danh sách các điểm cần vẽ
    scale=1                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

Max Abs Q ban đầu: 0.3233. Đang chuẩn hóa để vẽ...


In [ ]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='FPT',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_FPT,  # Dữ liệu giá
    actions=actions_FPT,             # Hành động thực tế
    q_history=q_values_history_FPT,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_FPT,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

Max Abs Q ban đầu: 0.1287. Đang chuẩn hóa để vẽ...


In [ ]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='GAS',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_GAS,  # Dữ liệu giá
    actions=actions_GAS,             # Hành động thực tế
    q_history=q_values_history_GAS,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_GAS,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

IndexError: list index out of range

In [300]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
fig_hpg = plot_agent_reasoning_minimalist(
    ticker='HPG',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_HPG,  # Dữ liệu giá
    actions=actions_HPG,             # Hành động thực tế
    q_history=q_values_history_HPG,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_HPG,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)
output_path_hpg = 'D:\\nckh\\SARSA_FinancialRL\\SARSA_FinancialRL\\Graph+Pic\\RDX_HPG_ucb.png'
fig_hpg.write_image(output_path_hpg, format='png', width=1400, height=900, scale=1)
print(f"✓ Đã xuất hình ảnh HPG tại: {output_path_hpg}")

# Hiển thị figure
fig_hpg.show()

Max Abs Q ban đầu: 0.2731. Đang chuẩn hóa để vẽ...
✓ Đã xuất hình ảnh HPG tại: D:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\Graph+Pic\RDX_HPG_ucb.png


In [ ]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='SSI',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_SSI,  # Dữ liệu giá
    actions=actions_SSI,             # Hành động thực tế
    q_history=q_values_history_SSI,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_SSI,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

IndexError: list index out of range

In [ ]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='VCB',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_VCB,  # Dữ liệu giá
    actions=actions_VCB,             # Hành động thực tế
    q_history=q_values_history_VCB,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_VCB,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

Max Abs Q ban đầu: 0.2602. Đang chuẩn hóa để vẽ...


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# 1. Cấu hình dữ liệu cho 6 Agent
# Đảm bảo bạn đã load đủ dữ liệu cho các biến này trước đó
agents_data = {
    'ACB': (df_processed_test_ACB, actions_ACB, q_values_history_ACB, index_list_ACB),
    'FPT': (df_processed_test_FPT, actions_FPT, q_values_history_FPT, index_list_FPT),
    'GAS': (df_processed_test_GAS, actions_GAS, q_values_history_GAS, index_list_GAS),
    'HPG': (df_processed_test_HPG, actions_HPG, q_values_history_HPG, index_list_HPG),
    'SSI': (df_processed_test_SSI, actions_SSI, q_values_history_SSI, index_list_SSI),
    'VCB': (df_processed_test_VCB, actions_VCB, q_values_history_VCB, index_list_VCB),
}

# 2. Tạo danh sách các Figure
figures = {}
print("Đang khởi tạo các biểu đồ...")

for ticker, (df, acts, q_hist, idxs) in agents_data.items():
    # Gọi hàm (nhớ là hàm này phải return fig nhé)
    fig = plot_agent_reasoning_minimalist(
        ticker=ticker,
        df_price=df,
        actions=acts,
        q_history=q_hist,
        critical_indices=idxs,
        scale=50 # Hoặc 10 tùy bạn chỉnh
    )
    figures[ticker] = fig
    print(f"- Đã vẽ xong {ticker}")

print("Hoàn tất!")

Đang khởi tạo các biểu đồ...
Max Abs Q ban đầu: 0.2591. Đang chuẩn hóa để vẽ...
- Đã vẽ xong ACB
Max Abs Q ban đầu: 0.1287. Đang chuẩn hóa để vẽ...
- Đã vẽ xong FPT
Max Abs Q ban đầu: 0.0873. Đang chuẩn hóa để vẽ...
- Đã vẽ xong GAS
Max Abs Q ban đầu: 0.1495. Đang chuẩn hóa để vẽ...
- Đã vẽ xong HPG
Max Abs Q ban đầu: 0.1012. Đang chuẩn hóa để vẽ...
- Đã vẽ xong SSI
Max Abs Q ban đầu: 0.2602. Đang chuẩn hóa để vẽ...
- Đã vẽ xong VCB
Hoàn tất!


In [ ]:
# Tạo danh sách các Widget Output (mỗi tab là một Output)
outputs = []
titles = []

for ticker, fig in figures.items():
    out = widgets.Output()
    with out:
        # Hiển thị biểu đồ trong container này
        fig.show()
    outputs.append(out)
    titles.append(ticker)

# Tạo Widget Tab
tabs = widgets.Tab(children=outputs)

# Đặt tên cho từng Tab
for i, title in enumerate(titles):
    tabs.set_title(i, title)

# Hiển thị giao diện tổng hợp
print("\n=== BẢNG ĐIỀU KHIỂN TỔNG HỢP (NHẤN VÀO TAB ĐỂ XEM) ===")
display(tabs)


=== BẢNG ĐIỀU KHIỂN TỔNG HỢP (NHẤN VÀO TAB ĐỂ XEM) ===


In [ ]:
# Gỡ bản hiện tại và cài bản 0.1.0post1
!pip uninstall -y kaleido
!pip install "kaleido==0.1.0post1"

  Using cached kaleido-0.1.0.post1-py2.py3-none-win_amd64.whl.metadata (15 kB)
Using cached kaleido-0.1.0.post1-py2.py3-none-win_amd64.whl (56.0 MB)


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\Admin\\anaconda3\\Lib\\site-packages\\kaleido\\executable\\version'
Consider using the `--user` option or check the permissions.



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

def plot_all_agents_combined(agents_data, scale=50):
    """
    Vẽ 6 Agent trong 1 khung hình lớn (Grid 3x2).
    Mỗi ô lưới chứa 2 biểu đồ con (Giá trên, Q-Bar dưới).
    """
    
    # Danh sách các mã (Keys)
    tickers = list(agents_data.keys())
    num_agents = len(tickers)
    
    # Cấu hình lưới: 6 hàng (3 hàng agent * 2 subplots/agent), 2 cột
    # row_heights: Phân bổ chiều cao cho Giá (lớn hơn) và Bar (nhỏ hơn)
    # Pattern: [Giá, Bar, Giá, Bar, Giá, Bar]
    row_specs = [0.20, 0.13] * 3 # Tổng xấp xỉ 1.0
    
    # Tiêu đề cho từng Agent (chỉ hiện ở biểu đồ giá)
    subplot_titles = []
    for i in range(0, 6, 2): # Duyệt theo cặp agent: (0,1), (2,3), (4,5)
        # Hàng hiện tại có 2 agent (trái và phải)
        idx_left = i 
        idx_right = i + 1
        
        title_left = f"<b>Agent: {tickers[idx_left]}</b>" if idx_left < num_agents else ""
        title_right = f"<b>Agent: {tickers[idx_right]}</b>" if idx_right < num_agents else ""
        
        # Thêm tiêu đề cho hàng Price
        subplot_titles.extend([title_left, title_right])
        # Hàng Bar không cần tiêu đề
        subplot_titles.extend(["", ""])

    # Tạo Figure lớn
    fig = make_subplots(
        rows=6, cols=2,
        row_heights=[0.6, 0.4, 0.6, 0.4, 0.6, 0.4], # Tỉ lệ chiều cao Price vs Bar
        vertical_spacing=0.06,
        horizontal_spacing=0.05,
        subplot_titles=subplot_titles,
        specs=[[{"secondary_y": False}, {"secondary_y": False}]] * 6
    )

    print("Đang tổng hợp dữ liệu và vẽ...")

    # --- VÒNG LẶP QUA TỪNG AGENT ---
    for i, ticker in enumerate(tickers):
        # Lấy dữ liệu
        df_price, actions, q_history, critical_indices = agents_data[ticker]
        
        # Xác định vị trí trong lưới (Row, Col)
        # i = 0 -> Row 1,2 Col 1
        # i = 1 -> Row 1,2 Col 2
        # i = 2 -> Row 3,4 Col 1
        col_idx = (i % 2) + 1
        base_row = (i // 2) * 2 + 1 # Row bắt đầu của Agent (1, 3, 5)
        row_price = base_row
        row_bar = base_row + 1
        
        # --- XỬ LÝ DỮ LIỆU (Copy từ hàm Minimalist) ---
        all_critical_q = np.array([q_history[t] for t in critical_indices])
        max_abs_q = np.max(np.abs(all_critical_q)) if len(all_critical_q) > 0 else 1
        if max_abs_q == 0: max_abs_q = 1

        buy_x, buy_y = [], []
        sell_x, sell_y = [], []
        hold_x, hold_y = [], []
        
        x_dates_hidden, x_actions = [], []
        y_profit, y_risk, y_pos, y_stab = [], [], [], []
        
        sel_grp, sel_act, sel_y = [], [], []
        tick_vals, tick_text = [], []

        for j, t in enumerate(critical_indices, start=1):  # Bắt đầu đếm từ 1
            date_str = df_price.iloc[t]['time']
            price = df_price.iloc[t]['close']
            act = actions[t]
            
            tick_vals.append(date_str)
            tick_text.append(date_str)
            
            if act > 0: buy_x.append(date_str); buy_y.append(price)
            elif act < 0: sell_x.append(date_str); sell_y.append(price)
            else: hold_x.append(date_str); hold_y.append(price)
            
            q_mat = (q_history[t] / max_abs_q) * scale
            grp_id = f"{j}" # ID nhóm bắt đầu từ 1
            
            compare_set = [("Sell", 0), ("Hold", 5), ("Buy", 10)]
            
            for label, idx in compare_set:
                vec = q_mat[idx]
                x_dates_hidden.append(grp_id)
                x_actions.append(label)
                y_profit.append(vec[0]); y_risk.append(vec[1])
                y_pos.append(vec[2]); y_stab.append(vec[3])
                
                is_selected = False
                if (act < 0 and idx == 0) or (act == 0 and idx == 5) or (act > 0 and idx == 10):
                    is_selected = True
                
                if is_selected:
                    sel_grp.append(grp_id)
                    sel_act.append(label)
                    pos_h = sum([v for v in vec if v > 0])
                    sel_y.append((pos_h if pos_h > 0 else 0) + scale * 0.05)

        # --- VẼ LÊN SUBPLOTS ---
        # Chỉ hiện legend cho agent đầu tiên
        show_legend = (i == 0)
        
        # 1. PRICE CHART
        fig.add_trace(go.Scatter(x=df_price['time'], y=df_price['close'], mode='lines', line=dict(color='black', width=1), showlegend=False), row=row_price, col=col_idx)
        fig.add_trace(go.Scatter(x=buy_x, y=buy_y, mode='markers', marker=dict(symbol='triangle-up', size=10, color='green'), name='Buy', showlegend=show_legend, legendgroup='act'), row=row_price, col=col_idx)
        fig.add_trace(go.Scatter(x=sell_x, y=sell_y, mode='markers', marker=dict(symbol='triangle-down', size=10, color='red'), name='Sell', showlegend=show_legend, legendgroup='act'), row=row_price, col=col_idx)
        fig.add_trace(go.Scatter(x=hold_x, y=hold_y, mode='markers', marker=dict(symbol='circle', size=8, color='#f1c40f'), name='Hold', showlegend=show_legend, legendgroup='act'), row=row_price, col=col_idx)

        # 2. BAR CHART
        for name, y_d, color in zip(['Profit', 'Risk', 'Trend', 'Stability'], [y_profit, y_risk, y_pos, y_stab], ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']):
            fig.add_trace(go.Bar(x=[x_dates_hidden, x_actions], y=y_d, name=name, marker_color=color, showlegend=show_legend, legendgroup='comp'), row=row_bar, col=col_idx)
            
        # Marker Selection
        fig.add_trace(go.Scatter(x=[sel_grp, sel_act], y=sel_y, mode='markers', marker=dict(symbol='triangle-down', size=10, color='black'), name='Agent Choice', showlegend=show_legend, legendgroup='choice'), row=row_bar, col=col_idx)

        # --- FORMAT TRỤC CHO TỪNG Ô ---
        # Trục X Price: Chỉ hiện tick tại ngày critical
        fig.update_xaxes(tickmode='array', tickvals=tick_vals, ticktext=['']*len(tick_vals), showgrid=True, row=row_price, col=col_idx) # Ẩn text ngày ở biểu đồ trên cho đỡ rối
        
        # Trục X Bar: Hiện ngày + hành động
        # Mẹo: Để hiển thị ngày tháng ở trục dưới cùng, ta dùng title hoặc ticktext tùy biến
        fig.update_xaxes(
            type='multicategory', 
            tickangle=-90, 
            showticklabels=True, 
            title_text=f"Critical Points ({len(critical_indices)})",
            title_font=dict(size=10),
            row=row_bar, col=col_idx
        )
        
        # Trục Y
        fig.update_yaxes(showgrid=True, gridcolor='#eee', row=row_price, col=col_idx)
        fig.update_yaxes(showgrid=True, gridcolor='#eee', row=row_bar, col=col_idx)
        
        # Đường 0
        fig.add_hline(y=0, line_color="black", line_width=1, row=row_bar, col=col_idx)

    # --- THÊM CÁC TRACE DUMMY ĐỂ ĐẢM BẢO LEGEND ĐẦY ĐỦ (SAU KHI VẼ TẤT CẢ) ---
    # Nếu agent đầu tiên không có Hold, thêm trace dummy cho Hold
    # Thêm vào vị trí cuối cùng để không ảnh hưởng đến ACB
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle', size=8, color='#f1c40f'),
        name='Hold', showlegend=True, legendgroup='act'
    ), row=6, col=2)

    # --- 3. LAYOUT CHUNG ---
    fig.update_layout(
        height=2000, 
        width=1900,  
        plot_bgcolor='white',
        barmode='relative',
        bargap=0.3, bargroupgap=0.05,
        # title_text="TỔNG HỢP PHÂN TÍCH QUYẾT ĐỊNH CỦA 6 AGENT (ACROSS-MARKET ANALYSIS)",
        title_font=dict(size=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.005, xanchor="center", x=0.5, bgcolor='rgba(255,255,255,0.8)'),
        margin=dict(t=100, b=50, l=50, r=50)
    )

    # SỬA ĐOẠN CUỐI CÙNG NÀY:
    # Thay vì chỉ fig.show(), hãy trả về fig
    return fig 

# --- LỆNH CHẠY VÀ LƯU ẢNH ---

# 1. Tạo đối tượng biểu đồ
fig_final = plot_all_agents_combined(agents_data, scale=1)

# 2. Hiển thị (để xem trước)
fig_final.show()

# 3. Lưu thành file PNG
print("Đang xuất file ảnh PNG (có thể mất vài giây)...")
fig_final.write_image("Tong_Hop_6_Agents_Full_phase_1.png", width=1900, height=2000, scale=1)
print("✅ Đã lưu xong: Tong_Hop_6_Agents_Full.png")

Đang tổng hợp dữ liệu và vẽ...


Đang xuất file ảnh PNG (có thể mất vài giây)...
✅ Đã lưu xong: Tong_Hop_6_Agents_Full.png


In [ ]:
# Xuất biểu đồ cho agent ACB ra file PNG
fig_acb = plot_agent_reasoning_minimalist(
    ticker='ACB',
    df_price=df_processed_test_ACB,
    actions=actions_ACB,
    q_history=q_values_history_ACB,
    critical_indices=index_list_ACB,
    scale=1
)

# Lưu file PNG
output_path_acb = 'D:\\nckh\\SARSA_FinancialRL\\SARSA_FinancialRL\\Graph+Pic\\RDX_ACB_2013_2017.png'
fig_acb.write_image(output_path_acb, format='png', width=1400, height=900, scale=1)
print(f"✓ Đã xuất hình ảnh ACB tại: {output_path_acb}")

# Hiển thị figure
fig_acb.show()

Max Abs Q ban đầu: 0.2591. Đang chuẩn hóa để vẽ...
✓ Đã xuất hình ảnh ACB tại: D:\nckh\SARSA_FinancialRL\SARSA_FinancialRL\Graph+Pic\RDX_ACB_2013_2017.png
